## 0. Install Dependencies

Run this cell first, then **restart the kernel**, then run all remaining cells.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("Installation done.")


# Payer Policy PA Parameter Extraction Pipeline

End-to-end RAG pipeline that extracts **12 Prior Authorization parameters** per brand from PsO payer policy PDFs.  
Uses a dual-model Groq setup — 8B for brand detection, 70B for parameter extraction — with hybrid BM25 + dense retrieval, RRF fusion, and cross-encoder reranking.

| Step | Description |
|------|-------------|
| 0 | Install Dependencies |
| 1 | Configuration and Imports |
| 2 | LLM Client (Groq · 8B + 70B · RateLimiter · in-memory cache) |
| 3 | PDF to Markdown (PyMuPDF + Docling, GPU-safe) |
| 4 | Chunking and Vector Store (700/100 · BM25 + BGE-384 + reranker) |
| 5 | Brand Detection (8B · hybrid search · anchor chunk IDs) |
| 6 | Parameter Extraction and Access Score (70B · two-tier retrieval) |
| 7 | Run Pipeline |
| 8 | Execute |


## 1. Configuration and Imports

In [ ]:
# torch must be imported and fully initialised BEFORE docling/transformers
# to prevent "AttributeError: module torch has no attribute _utils"
import torch
import torch._utils
import torch.utils

import csv
import hashlib
import json
import logging
import os
import re
import shutil
import tempfile
import time
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from tqdm.auto import tqdm
import fitz
import chromadb
from chromadb.config import Settings as ChromaSettings
import numpy as np
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from groq import Groq
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from tenacity import (
    retry, retry_if_exception_type, stop_after_attempt, wait_exponential,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
logging.getLogger("chromadb.telemetry").setLevel(logging.CRITICAL)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Kaggle: Add-ons > Secrets > add GROQ_API_KEY, enable "Attach to session"
# Local:  export GROQ_API_KEY=gsk_...
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
assert GROQ_API_KEY, "Set GROQ_API_KEY in Kaggle Secrets (or export GROQ_API_KEY=...)"
print(f"Groq key set: {bool(GROQ_API_KEY)}")

_HERE        = Path(".").resolve()
PDF_DIR      = _HERE / "pdfs" / "Sample_PsO_ADS_Track"
MARKDOWN_DIR = _HERE / "markdown_output"
OUTPUT_CSV   = _HERE / "submission.csv"
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE = _HERE / "model_cache"
MODEL_CACHE.mkdir(exist_ok=True)

# --- Embeddings: 384-dim bge-small (faster + lighter than bge-base) --------
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"
RERANK_MODEL  = "BAAI/bge-reranker-v2-m3"
COLLECTION    = "payer_policy"
CHUNK_SIZE    = 700
CHUNK_OVERLAP = 100
RRF_K         = 60
PARAM_TOP_K   = 4   # max chunks passed to 70B (keeps each call < ~2500 tokens)
HEADER_RATIO  = 0.07
FOOTER_RATIO  = 0.07

# --- Groq models -------------------------------------------------------------
GROQ_MODEL_8B  = "llama-3.1-8b-instant"     # brand detection
GROQ_MODEL_70B = "llama-3.3-70b-versatile"  # parameter extraction

# Free-tier limits (we stay 1 request + 5% tokens below these)
_GROQ_LIMITS: Dict[str, Dict[str, int]] = {
    "8b":  {"rpm": 30, "tpm": 131_072},
    "70b": {"rpm": 30, "tpm":  12_000},
}

PARAMS = [
    "Age",
    "Step Therapy Requirements Documented in Policy",
    "Number of Steps through Brands",
    "Number of Steps through Generic",
    "Step through-Phototherapy",
    "TB Test required",
    "Initial Authorization Duration(in-months)",
    "Reauthorization Duration(in-months)",
    "Reauthorization Required",
    "Reauthorization Requirements Documented in Policy",
    "Specialist Types",
    "Quantity Limits",
]
CSV_COLUMNS = ["filename", "brand"] + PARAMS + ["access_score"]

print(f"PDF dir      : {PDF_DIR}")
print(f"Output CSV   : {OUTPUT_CSV}")
print(f"Chunk size   : {CHUNK_SIZE}  overlap={CHUNK_OVERLAP}")
print(f"Embed model  : {EMBED_MODEL}  (384-dim)")
print(f"8B model     : {GROQ_MODEL_8B}")
print(f"70B model    : {GROQ_MODEL_70B}")


## 2. LLM Client

Groq-only, **two-model** design:

| Role | Model | Groq limit |
|------|-------|------------|
| Brand detection | `llama-3.1-8b-instant` | 131K TPM |
| Param extraction | `llama-3.3-70b-versatile` | 12K TPM |

`RateLimiter` tracks requests and tokens in a 60-second sliding window,
staying 1 request and 5% tokens below the free-tier limits.
In-memory cache (MD5 key) avoids duplicate API calls.


In [ ]:
# ---------------------------------------------------------------------------
# Module-level response cache  {md5_key -> response_text}
# ---------------------------------------------------------------------------
_llm_cache: Dict[str, str] = {}


def _cache_key(messages: List[Dict], model: str) -> str:
    payload = json.dumps({"m": messages, "model": model}, sort_keys=True)
    return hashlib.md5(payload.encode()).hexdigest()


# ---------------------------------------------------------------------------
# Rate limiter — sliding 60-second window for RPM + TPM
# ---------------------------------------------------------------------------
class RateLimiter:
    """
    Stays just below Groq free-tier limits:
      - max_rpm = rpm - 1  (one request below the cap)
      - max_tpm = tpm * 0.95  (5% buffer for token estimation error)
    """
    def __init__(self, rpm: int, tpm: int, buffer: float = 0.05):
        self._max_rpm = rpm - 1
        self._max_tpm = int(tpm * (1 - buffer))
        self._req_times: deque = deque()   # float timestamps
        self._tok_log:   deque = deque()   # (timestamp, int_tokens)

    def _purge(self) -> None:
        cutoff = time.time() - 60.0
        while self._req_times and self._req_times[0] < cutoff:
            self._req_times.popleft()
        while self._tok_log and self._tok_log[0][0] < cutoff:
            self._tok_log.popleft()

    def wait_if_needed(self, estimated_tokens: int) -> None:
        while True:
            self._purge()
            reqs = len(self._req_times)
            toks = sum(t for _, t in self._tok_log)
            if reqs < self._max_rpm and toks + estimated_tokens <= self._max_tpm:
                return
            oldest = min(
                self._req_times[0]  if self._req_times  else time.time(),
                self._tok_log[0][0] if self._tok_log    else time.time(),
            )
            sleep_for = max(0.5, oldest + 60.1 - time.time())
            log.info(
                "Rate limit — sleeping %.1fs  (rpm %d/%d, tpm %d/%d)",
                sleep_for, reqs, self._max_rpm, toks, self._max_tpm,
            )
            time.sleep(sleep_for)

    def record(self, tokens_used: int) -> None:
        now = time.time()
        self._req_times.append(now)
        self._tok_log.append((now, tokens_used))


# ---------------------------------------------------------------------------
# LLM Client
# ---------------------------------------------------------------------------
class LLMClient:
    """
    Groq client for 8B (brand detection) or 70B (param extraction).

    Usage:
        llm_8b  = LLMClient("8b")
        llm_70b = LLMClient("70b")
        text = llm_8b.complete([{"role": "user", "content": "hi"}])
    """

    def __init__(self, model_size: str = "8b", fallback: Optional["LLMClient"] = None):
        if model_size not in _GROQ_LIMITS:
            raise ValueError(f"model_size must be '8b' or '70b', got '{model_size}'")
        self.model_size = model_size
        self.model      = GROQ_MODEL_8B if model_size == "8b" else GROQ_MODEL_70B
        limits          = _GROQ_LIMITS[model_size]
        self.limiter    = RateLimiter(limits["rpm"], limits["tpm"])
        self._client    = Groq(api_key=GROQ_API_KEY)
        self.fallback   = fallback
        self._degraded  = False  # set True after first fallback; routes all calls to fallback
        log.info("LLMClient ready — model=%s", self.model)

    def complete(
        self,
        messages:      List[Dict],
        temperature:   float = 0.0,
        max_tokens:    int   = 1024,
        use_llm_cache: bool  = True,
    ) -> str:
        # If degraded (rate limit hit before), skip straight to fallback
        if self._degraded and self.fallback:
            return self.fallback.complete(messages, temperature, max_tokens, use_llm_cache)

        cache_key = _cache_key(messages, self.model)
        if use_llm_cache and cache_key in _llm_cache:
            log.debug("Cache hit [%s]", cache_key[:8])
            return _llm_cache[cache_key]

        # Estimate tokens: prompt chars / 4 + output budget
        estimated = len(json.dumps(messages)) // 4 + max_tokens
        self.limiter.wait_if_needed(estimated)

        log.debug("API call [%s]  est_tokens=%d", cache_key[:8], estimated)
        try:
            response = self._call(messages, temperature, max_tokens)
        except Exception as exc:
            if self.fallback and ("rate" in str(exc).lower() or "429" in str(exc)):
                log.warning(
                    "Rate limit on %s — falling back to %s for remainder of run",
                    self.model, self.fallback.model,
                )
                self._degraded = True
                return self.fallback.complete(messages, temperature, max_tokens, use_llm_cache)
            raise

        # Record approximated actual usage
        actual = len(json.dumps(messages)) // 4 + len(response) // 4
        self.limiter.record(actual)

        if use_llm_cache:
            _llm_cache[cache_key] = response
        return response

    @retry(
        stop=stop_after_attempt(4),
        wait=wait_exponential(multiplier=2, min=4, max=60),
        retry=retry_if_exception_type(Exception),
        reraise=True,
    )
    def _call(self, messages: List[Dict], temperature: float, max_tokens: int) -> str:
        resp = self._client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return resp.choices[0].message.content.strip()

    @property
    def cache_size(self) -> int:
        return len(_llm_cache)

    def clear_cache(self) -> None:
        _llm_cache.clear()
        log.info("LLM cache cleared")


print("RateLimiter + LLMClient defined.")


## 3. PDF to Markdown

PyMuPDF cleans the PDF in memory (headers/footers/links/credentials), then Docling converts it to Markdown with table detection (forced CPU).

In [ ]:
HEADER_RATIO = 0.07
FOOTER_RATIO = 0.07

_CRED_PATTERNS = [
    re.compile(r"(username|user[\s_-]*name|login|user[\s_-]*id)\s*[:=]\s*\S+", re.I),
    re.compile(r"(password|passwd|pwd)\s*[:=]\s*\S+", re.I),
    re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}"),
]


def _redact_zone(page: fitz.Page, zone: fitz.Rect) -> None:
    for b in page.get_text("blocks", clip=zone):
        page.add_redact_annot(fitz.Rect(b[:4]), fill=(1, 1, 1))


def _redact_credentials(page: fitz.Page) -> None:
    for block in page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE).get("blocks", []):
        if block.get("type") != 0:
            continue
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                if any(p.search(span.get("text", "")) for p in _CRED_PATTERNS):
                    page.add_redact_annot(fitz.Rect(span["bbox"]), fill=(1, 1, 1))



_REF_PATTERNS = [
    re.compile(r'^\s*references\s*$', re.I),
    re.compile(r'the above policy is based on the following references', re.I),
]


def _strip_references(doc: fitz.Document) -> None:
    """
    Find the first page containing a References heading and redact everything
    from that heading to the end of the document.
    Pages after the references page are deleted entirely.
    """
    ref_page_idx = None
    ref_y = None

    for page_num in range(len(doc)):
        page = doc[page_num]
        blocks = page.get_text("blocks")
        for b in blocks:
            text = b[4].strip()
            if any(p.search(text) for p in _REF_PATTERNS):
                ref_page_idx = page_num
                ref_y = b[1]  # top y-coordinate of the references block
                break
        if ref_page_idx is not None:
            break

    if ref_page_idx is None:
        return

    # Redact from ref_y to bottom of the references page
    ref_page = doc[ref_page_idx]
    h = ref_page.rect.height
    w = ref_page.rect.width
    ref_page.add_redact_annot(fitz.Rect(0, ref_y, w, h), fill=(1, 1, 1))
    ref_page.apply_redactions()

    # Delete all pages after the references page
    for page_num in range(len(doc) - 1, ref_page_idx, -1):
        doc.delete_page(page_num)


def _clean_doc(doc: fitz.Document) -> None:
    for page in doc:
        h, w = page.rect.height, page.rect.width
        _redact_zone(page, fitz.Rect(0, 0, w, h * HEADER_RATIO))
        _redact_zone(page, fitz.Rect(0, h * (1 - FOOTER_RATIO), w, h))
        for link in page.get_links():
            page.delete_link(link)
        _redact_credentials(page)
        page.apply_redactions()


def _hide_cuda():
    """Return current CUDA_VISIBLE_DEVICES value and set it to empty string."""
    bak = os.environ.get("CUDA_VISIBLE_DEVICES")
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    return bak


def _restore_cuda(bak: Optional[str]) -> None:
    """Restore CUDA_VISIBLE_DEVICES to its original value."""
    if bak is None:
        os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    else:
        os.environ["CUDA_VISIBLE_DEVICES"] = bak




def build_converter() -> DocumentConverter:
    """
    Build the Docling DocumentConverter with CUDA hidden.
    This prevents RT-DETRv2 from attempting to load the MultiScaleDeformableAttention
    CUDA kernel (not compiled on Kaggle), which would otherwise print the
    "Could not load custom kernel" warning dozens of times.
    CUDA is restored after the converter is built so embeddings still use GPU.
    """
    bak = _hide_cuda()
    try:
        opts = PdfPipelineOptions()
        opts.do_ocr = False
        opts.do_table_structure = True
        opts.table_structure_options.do_cell_matching = True

        try:
            from docling.datamodel.pipeline_options import AcceleratorOptions
            from docling.datamodel.accelerator_options import AcceleratorDevice
            opts.accelerator_options = AcceleratorOptions(
                num_threads=4, device=AcceleratorDevice.CPU
            )
        except ImportError:
            pass

        return DocumentConverter(
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
        )
    finally:
        _restore_cuda(bak)


def convert_pdf(
    pdf_path:  Path,
    md_dir:    Path,
    converter: DocumentConverter,
) -> Optional[Path]:
    """
    Clean pdf_path in memory and convert to Markdown via Docling.
    CUDA is hidden during conversion to prevent the "Cannot copy out of meta
    tensor" crash on Kaggle GPU runtimes (RT-DETRv2 uses .to(device) on
    uninitialised meta tensors when a GPU is detected).
    Returns the Path to the written .md file, or None on failure.
    """
    md_out = md_dir / (pdf_path.stem + ".md")
    if md_out.exists():
        log.info("[skip]    %s — markdown already exists", pdf_path.name)
        return md_out

    bak      = _hide_cuda()
    tmp_path = None
    try:
        doc = fitz.open(str(pdf_path))
        _clean_doc(doc)
        _strip_references(doc)

        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
            tmp_path = Path(tmp.name)
            doc.save(str(tmp_path), garbage=4, deflate=True)
        doc.close()

        result   = converter.convert(str(tmp_path))
        tmp_path.unlink(missing_ok=True)
        tmp_path = None

        document = result.document
        markdown = document.export_to_markdown()

        n_tables = 0
        try:
            from docling_core.types.doc import DocItemLabel
            n_tables = sum(
                1 for item, _ in document.iterate_items()
                if getattr(item, "label", None) == DocItemLabel.TABLE
            )
        except Exception:
            n_tables = markdown.count("\n|")

        md_dir.mkdir(parents=True, exist_ok=True)
        md_out.write_text(markdown, encoding="utf-8")
        log.info("[done]    %s  | tables: %d | chars: %d",
                 pdf_path.name, n_tables, len(markdown))
        return md_out

    except Exception as exc:
        log.error("[fail]    %s — %s", pdf_path.name, exc)
        if tmp_path:
            Path(tmp_path).unlink(missing_ok=True)
        return None

    finally:
        _restore_cuda(bak)


print("PDF to Markdown functions defined.")


## 4. Chunking and Vector Store

Recursive character splitting (**700 chars / 100 overlap**),
ChromaDB storage, BM25 + 384-dim cosine + RRF + cross-encoder reranker.


In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------



# ---------------------------------------------------------------------------
# Data model
# ---------------------------------------------------------------------------
@dataclass
class Chunk:
    chunk_id: str
    text: str
    columns: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)


# ---------------------------------------------------------------------------
# Recursive text splitter
# ---------------------------------------------------------------------------
_SEPARATORS = ["\n## ", "\n### ", "\n#### ", "\n\n", "\n", ". ", " ", ""]


def _merge_splits(splits: List[str], sep: str, size: int, overlap: int) -> List[str]:
    """
    Merge atomic splits into chunks <= size chars.
    Sliding-window: flush when the next split would exceed size,
    pop from the front until retained context <= overlap.
    """
    chunks: List[str] = []
    window: List[str] = []
    window_len = 0
    sep_len = len(sep)

    def flush() -> None:
        if not window:
            return
        chunk = sep.join(window)
        if len(chunk) <= size:
            chunks.append(chunk)
        else:
            step = max(1, size - overlap)
            for i in range(0, len(chunk), step):
                piece = chunk[i: i + size]
                if piece.strip():
                    chunks.append(piece)

    for s in splits:
        s_len = len(s)
        add_len = s_len + (sep_len if window else 0)

        if window_len + add_len > size:
            flush()
            while window and window_len > overlap:
                removed = window.pop(0)
                window_len -= len(removed) + (sep_len if window else 0)

        window.append(s)
        window_len += s_len + (sep_len if len(window) > 1 else 0)

    flush()
    return chunks


def _recursive_split(text: str, size: int = CHUNK_SIZE,
                     overlap: int = CHUNK_OVERLAP,
                     seps: List[str] = _SEPARATORS) -> List[str]:
    """
    Recursively split text using separators from coarsest to finest,
    then merge into chunks of at most `size` chars with `overlap`.
    """
    if not text.strip():
        return []
    if len(text) <= size:
        return [text.strip()]

    sep = None
    remaining_seps: List[str] = []
    for i, s in enumerate(seps):
        if s == "" or s in text:
            sep = s
            remaining_seps = seps[i + 1:]
            break

    if sep is None:
        return [text.strip()]

    if sep == "":
        step = max(1, size - overlap)
        return [text[i: i + size].strip()
                for i in range(0, len(text), step)
                if text[i: i + size].strip()]

    flat: List[str] = []
    for p in text.split(sep):
        p = p.strip()
        if not p:
            continue
        if len(p) <= size:
            flat.append(p)
        else:
            flat.extend(_recursive_split(p, size, overlap, remaining_seps))

    join_sep = sep.strip() or " "
    return [c.strip() for c in _merge_splits(flat, join_sep, size, overlap) if c.strip()]


# ---------------------------------------------------------------------------
# Table helpers
# ---------------------------------------------------------------------------
_TABLE_RE = re.compile(
    r"(\|[^\n]+\|\n\|[-| :]+\|\n(?:\|[^\n]+\|\n)*)",
    re.MULTILINE,
)
_HEADER_RE = re.compile(r"^#{1,4}\s+(.+)", re.MULTILINE)


def _extract_columns(table_text: str) -> List[str]:
    first_line = table_text.strip().splitlines()[0]
    return [c.strip() for c in first_line.split("|") if c.strip()]


def _split_table(table_text: str, columns: List[str],
                 size: int, meta: Dict) -> List[Chunk]:
    """Split an oversized table into row-batches, each prefixed with the header row."""
    lines = table_text.strip().splitlines()
    if len(lines) < 3:
        return []

    header_block = lines[0] + "\n" + lines[1] + "\n"
    chunks: List[Chunk] = []
    buf = header_block

    for row in lines[2:]:
        candidate = buf + row + "\n"
        if len(candidate) <= size:
            buf = candidate
        else:
            if buf.strip() != header_block.strip():
                chunks.append(Chunk(text=buf.strip(), columns=columns,
                                    metadata=meta, chunk_id=""))
            buf = header_block + row + "\n"

    if buf.strip() and buf.strip() != header_block.strip():
        chunks.append(Chunk(text=buf.strip(), columns=columns,
                            metadata=meta, chunk_id=""))
    return chunks


# ---------------------------------------------------------------------------
# Markdown chunker
# ---------------------------------------------------------------------------
def chunk_markdown(md_text: str, pdf_name: str) -> List[Chunk]:
    """Parse one markdown document into Chunk objects."""
    chunks: List[Chunk] = []
    current_header = "Introduction"
    idx = 0

    def _next_id() -> str:
        nonlocal idx
        cid = f"{Path(pdf_name).stem}_{idx:04d}"
        idx += 1
        return cid

    segments: List[Dict] = []
    last_end = 0
    for m in _TABLE_RE.finditer(md_text):
        if m.start() > last_end:
            segments.append({"type": "prose", "text": md_text[last_end:m.start()]})
        segments.append({"type": "table", "text": m.group(0)})
        last_end = m.end()
    if last_end < len(md_text):
        segments.append({"type": "prose", "text": md_text[last_end:]})

    for seg in segments:
        # ---- TABLE --------------------------------------------------------
        if seg["type"] == "table":
            columns = _extract_columns(seg["text"])
            meta = {"table": True, "pdf": pdf_name, "header": current_header}
            if len(seg["text"]) <= CHUNK_SIZE:
                chunks.append(Chunk(
                    chunk_id=_next_id(),
                    text=seg["text"].strip(),
                    columns=columns,
                    metadata=meta,
                ))
            else:
                for c in _split_table(seg["text"], columns, CHUNK_SIZE, meta):
                    c.chunk_id = _next_id()
                    chunks.append(c)

        # ---- PROSE --------------------------------------------------------
        else:
            prose = seg["text"]
            for hdr in _HEADER_RE.finditer(prose):
                current_header = hdr.group(1).strip()

            for split in _recursive_split(prose):
                for hdr in _HEADER_RE.finditer(split):
                    current_header = hdr.group(1).strip()
                chunks.append(Chunk(
                    chunk_id=_next_id(),
                    text=split,
                    columns=[],
                    metadata={"table": False, "pdf": pdf_name, "header": current_header},
                ))

    return chunks


# ---------------------------------------------------------------------------
# ChromaDB store + hybrid search
# ---------------------------------------------------------------------------
class PolicyStore:
    def __init__(self, chroma_dir: Path = Path('./chroma_store'), embed_model: str = EMBED_MODEL,
                 rerank_model: str = RERANK_MODEL):
        log.info("Loading embedding model: %s", embed_model)
        self.encoder = SentenceTransformer(embed_model, device=DEVICE, cache_folder=str(MODEL_CACHE))
        log.info("Embedding dim: %d", self.encoder.get_sentence_embedding_dimension())

        log.info("Loading reranker: %s", rerank_model)
        self.reranker = CrossEncoder(rerank_model, device=DEVICE, cache_folder=str(MODEL_CACHE))

        self.client = chromadb.PersistentClient(
            path=str(chroma_dir),
            settings=ChromaSettings(anonymized_telemetry=False),
        )
        self.col = self.client.get_or_create_collection(
            name=COLLECTION,
            metadata={"hnsw:space": "cosine"},
        )
        log.info("Collection '%s' — %d docs stored", COLLECTION, self.col.count())

        self._bm25: Optional[BM25Okapi] = None
        self._bm25_ids: List[str] = []
        self._bm25_texts: List[str] = []

    def add_chunks(self, chunks: List[Chunk], batch_size: int = 64) -> int:
        existing = set(self.col.get(include=[])["ids"])
        new = [c for c in chunks if c.chunk_id not in existing]
        if not new:
            return 0

        for i in range(0, len(new), batch_size):
            batch = new[i: i + batch_size]
            texts = [c.text for c in batch]
            embeddings = self.encoder.encode(
                texts, batch_size=32, show_progress_bar=False,
                normalize_embeddings=True,
            ).tolist()
            self.col.add(
                ids=[c.chunk_id for c in batch],
                documents=texts,
                embeddings=embeddings,
                metadatas=[
                    {
                        **c.metadata,
                        "columns": "|".join(c.columns),
                        "table": str(c.metadata.get("table", False)),
                    }
                    for c in batch
                ],
            )

        self._bm25 = None
        log.info("Stored %d new chunks (%d already present)", len(new), len(existing))
        return len(new)

    def _ensure_bm25(self) -> None:
        if self._bm25 is not None:
            return
        result = self.col.get(include=["documents"])
        self._bm25_ids = result["ids"]
        self._bm25_texts = result["documents"]
        self._bm25 = BM25Okapi([t.lower().split() for t in self._bm25_texts])
        log.info("BM25 index built over %d docs", len(self._bm25_ids))

    def hybrid_search(self, query: str, top_k: int = 10,
                      rerank_candidates: int = 30) -> List[Dict[str, Any]]:
        """
        Three-stage retrieval:
          1. BM25 + dense cosine → RRF → top `rerank_candidates`
          2. bge-reranker-v2-m3 cross-encoder → re-scores each candidate
          3. Return top `top_k` by reranker score
        """
        self._ensure_bm25()
        n_candidates = min(rerank_candidates, max(self.col.count(), 1))

        # --- Stage 1a: BM25 sparse ---
        bm25_scores = self._bm25.get_scores(query.lower().split())
        sparse_ranks = np.argsort(bm25_scores)[::-1][:n_candidates].tolist()

        # --- Stage 1b: dense cosine ---
        q_vec = self.encoder.encode([query], normalize_embeddings=True).tolist()
        dense = self.col.query(
            query_embeddings=q_vec,
            n_results=n_candidates,
            include=["documents", "metadatas", "distances"],
        )
        dense_ids: List[str] = dense["ids"][0]

        # --- Stage 1c: RRF fusion ---
        rrf: Dict[str, float] = {}
        for rank, arr_idx in enumerate(sparse_ranks):
            cid = self._bm25_ids[arr_idx]
            rrf[cid] = rrf.get(cid, 0.0) + 1.0 / (RRF_K + rank + 1)
        for rank, cid in enumerate(dense_ids):
            rrf[cid] = rrf.get(cid, 0.0) + 1.0 / (RRF_K + rank + 1)

        candidate_ids = sorted(rrf, key=lambda x: rrf[x], reverse=True)[:n_candidates]

        # Fetch full text for all candidates
        fetched = self.col.get(ids=candidate_ids, include=["documents", "metadatas"])
        id_map = {
            cid: (doc, meta)
            for cid, doc, meta in zip(
                fetched["ids"], fetched["documents"], fetched["metadatas"]
            )
        }

        # --- Stage 2: rerank ---
        pairs = [(query, id_map[cid][0]) for cid in candidate_ids if cid in id_map]
        rerank_scores = self.reranker.predict(pairs)   # shape: (n_candidates,)

        ranked = sorted(
            zip(candidate_ids, rerank_scores),
            key=lambda x: x[1],
            reverse=True,
        )[:top_k]

        # --- Stage 3: build results ---
        return [
            {
                "chunk_id": cid,
                "text": id_map[cid][0],
                "columns": [c for c in id_map[cid][1].get("columns", "").split("|") if c],
                "metadata": {
                    "table":  id_map[cid][1].get("table") == "True",
                    "pdf":    id_map[cid][1].get("pdf", ""),
                    "header": id_map[cid][1].get("header", ""),
                },
                "rrf_score":    round(rrf.get(cid, 0.0), 6),
                "rerank_score": round(float(score), 4),
            }
            for cid, score in ranked if cid in id_map
        ]


# ---------------------------------------------------------------------------
# Entry point — processes ONE markdown file
# ---------------------------------------------------------------------------

print("Chunking and PolicyStore defined.")

## 5. Brand Detection  (8B)

Two-stage using `llama-3.1-8b-instant`:

1. **Hybrid search** for drug-list sections (`applicable drug list`, `formulary`, `biologics`) \
   Top chunks passed to 8B to identify PsO brand names.
2. **Anchor search** — for each detected brand, run a targeted hybrid search \
   and store the top chunk IDs as `anchor_chunk_ids` for use in param extraction.

Falls back to raw markdown truncation if no drug-list chunks are found.


In [ ]:
MAX_CHARS = 4000  
_DRUG_LIST_QUERIES = [
    "applicable drug list biologics PsO formulary covered medications",
    "preferred non-preferred biologic brand names plaque psoriasis",
    "step therapy biologic agents psoriasis prior authorization criteria",
]

_BRAND_PROMPT = """\
You are an expert at extracting structured prior authorization policy information from payer policy documents.

Your task is to identify all brands/products in this policy document that are relevant to plaque psoriasis (PsO).

Instructions:
1. Read the provided policy text carefully.
2. Identify all products listed in the Applicable Drug List or equivalent drug list section.
3. Determine whether the policy contains coverage criteria for plaque psoriasis (PsO).
4. Return every product/brand that is relevant to PsO extraction.
5. Include preferred/non-preferred status if explicitly stated.
6. Do not infer brands that are not explicitly listed.
7. If the policy has multiple indications, only identify brands relevant to PsO.

Return strict JSON only in this format:
{{
  "policy_has_pso": "Yes | No",
  "brands_relevant_to_pso": [
    {{
      "brand": "",
      "preferred_status": "Preferred | Non-preferred | Unspecified"
    }}
  ]
}}

Policy Text:
{policy_text}"""


def _truncate(text: str, max_chars: int = MAX_CHARS) -> str:
    if len(text) <= max_chars:
        return text
    head = int(max_chars * 0.67)
    tail = max_chars - head
    return text[:head] + "\n\n[...truncated...]\n\n" + text[-tail:]


def _parse_json(raw: str, pdf_name: str) -> Dict[str, Any]:
    try:
        start = raw.index("{")
        end   = raw.rindex("}") + 1
        return json.loads(raw[start:end])
    except (ValueError, json.JSONDecodeError):
        log.error("Brand JSON parse failed for %s:\n%s", pdf_name, raw[:300])
        return {"policy_has_pso": "No", "brands_relevant_to_pso": []}


def detect_brands(md_path: Path, store: "PolicyStore", llm: LLMClient) -> Dict[str, Any]:
    """
    Stage 1: Hybrid-search for drug-list chunks, pass to 8B -> brand list JSON.
    Stage 2: Per-brand anchor search -> stores anchor_chunk_ids.

    Returns:
        {
          "policy_has_pso": "Yes" | "No",
          "brands_relevant_to_pso": [
            {"brand": ..., "preferred_status": ..., "anchor_chunk_ids": [...]}
          ]
        }
    """
    pdf_name = md_path.stem + ".pdf"

    # Stage 1: find drug-list sections via hybrid search
    seen: set = set()
    context_chunks: List[Dict] = []
    for query in _DRUG_LIST_QUERIES:
        for r in store.hybrid_search(query, top_k=3):
            if r["metadata"]["pdf"] == pdf_name and r["chunk_id"] not in seen:
                seen.add(r["chunk_id"])
                context_chunks.append(r)

    if context_chunks:
        policy_text = "\n\n---\n\n".join(r["text"] for r in context_chunks)
        if len(policy_text) > MAX_CHARS:
            policy_text = policy_text[:MAX_CHARS]
        log.info(
            "Brand detection — %s  (%d chunks, %d chars)",
            pdf_name, len(context_chunks), len(policy_text),
        )
    else:
        log.warning("No drug-list chunks found — falling back to raw markdown")
        policy_text = _truncate(md_path.read_text(encoding="utf-8"))
        log.info("Brand detection — %s  (raw fallback, %d chars)", pdf_name, len(policy_text))

    messages = [{"role": "user", "content": _BRAND_PROMPT.format(policy_text=policy_text)}]
    raw      = llm.complete(messages, temperature=0.0, max_tokens=2048)
    result   = _parse_json(raw, pdf_name)

    # Stage 2: anchor chunk IDs for each detected brand
    brands = result.get("brands_relevant_to_pso", [])
    for brand in brands:
        brand_name = brand["brand"]
        anchor_ids: List[str] = []
        for r in store.hybrid_search(
            f"{brand_name} plaque psoriasis prior authorization step therapy criteria",
            top_k=3,
        ):
            if r["metadata"]["pdf"] == pdf_name:
                anchor_ids.append(r["chunk_id"])
        brand["anchor_chunk_ids"] = anchor_ids

    log.info(
        "  policy_has_pso=%s  brands=%d",
        result.get("policy_has_pso", "?"), len(brands),
    )
    return result


print("Brand detection defined.")


## 6. Parameter Extraction and Access Score  (70B)

Two-tier retrieval per brand using `llama-3.3-70b-versatile`:

1. **Anchor chunks** (from brand detection stage 2) -- guaranteed relevant hits
2. **Hybrid search top-3** -- fills remaining slots to a hard cap of **4 chunks** (~2500 tokens/call)

The built-in `RateLimiter` respects Groq's 12K TPM free-tier limit for the 70B model.

Access Score (1-100): weighted sum -- higher = easier access.

| Parameter | Weight |
|-----------|--------|
| Steps through Brands | 20 |
| Initial Auth Duration | 15 |
| TB Test required | 15 |
| Age | 10 |
| Steps through Generic | 10 |
| Step through-Phototherapy | 5 |
| Step Therapy text | 5 |
| Reauth Required | 5 |
| Reauth Duration | 5 |
| Specialist Types | 4 |
| Reauth Requirements text | 3 |
| Quantity Limits | 3 |


In [ ]:
# ---------------------------------------------------------------------------
# Prompt -- one brand at a time
# ---------------------------------------------------------------------------
_PARAM_PROMPT = """\
You are an expert in extracting structured prior authorization policy data from payer policy documents.

Extract 12 PsO-specific parameters for the brand below using ONLY the provided policy chunks.

BRAND:
  Name             : {brand_name}
  Preferred status : {preferred_status}

INSTRUCTIONS:
- Extract for plaque psoriasis (PsO) only. Ignore other indications.
- If moderate-to-severe and severe PsO are distinguished, use moderate-to-severe criteria only.
- Universal criteria that apply to all brands must be combined with brand-specific criteria using AND logic.
- If OR statements exist, choose the least restrictive valid path.
- Count only what is explicitly stated. Do not infer.
- Use "NA" for any value not mentioned, unless rules below specify otherwise.
- Output strict JSON only -- no explanation.

PARAMETERS:

1. Age: Age threshold for eligibility. Output "FDA labelled age" if only FDA labelling is mentioned. If two age groups listed, capture the youngest.

2. Step Therapy Requirements Documented in Policy: Full free-text of all step therapy language relevant to PsO for this brand (universal + brand-specific).

3. Number of Steps through Brands: Count of branded/biologic steps required before this brand is approved. Choose least restrictive OR path. Exclude phototherapy. "NA" if none.

4. Number of Steps through Generic: Count of non-biologic/generic/topical steps required. Exclude phototherapy. "NA" if none.

5. Step through-Phototherapy: "Yes" if phototherapy is a mandatory step. "No" if not required. "N/A" if no criteria at all.

6. TB Test required: "Y" if required. "N" if explicitly not required. "NA" if not mentioned.

7. Initial Authorization Duration(in-months): Numeric months. "Unspecified" if required but not stated numerically.

8. Reauthorization Duration(in-months): Numeric months. "Unspecified" if required but not stated numerically.

9. Reauthorization Required: "Yes" if reauth is documented. "No" if explicitly not required. "NA" otherwise.

10. Reauthorization Requirements Documented in Policy: Actual continuation/renewal criteria text for PsO.

11. Specialist Types: Specialist type(s) acceptable for prescribing/managing PsO treatment.

12. Quantity Limits: Only explicitly stated quantity limits. Do NOT extract dosage language. "NA" if not stated.

OUTPUT FORMAT -- strict JSON only:
{{
  "brand": "{brand_name}",
  "preferred_status": "{preferred_status}",
  "Age": "",
  "Step Therapy Requirements Documented in Policy": "",
  "Number of Steps through Brands": "",
  "Number of Steps through Generic": "",
  "Step through-Phototherapy": "",
  "TB Test required": "",
  "Initial Authorization Duration(in-months)": "",
  "Reauthorization Duration(in-months)": "",
  "Reauthorization Required": "",
  "Reauthorization Requirements Documented in Policy": "",
  "Specialist Types": "",
  "Quantity Limits": ""
}}

RELEVANT POLICY CHUNKS:
{chunks}"""

# ---------------------------------------------------------------------------
# Retrieval queries for common parameters (used as fallback fill)
# ---------------------------------------------------------------------------
_COMMON_QUERIES = [
    "step therapy prior authorization criteria plaque psoriasis PsO",
    "initial authorization duration months reauthorization renewal continuation",
    "TB test tuberculosis quantity limit specialist prescriber dermatologist",
]


# ---------------------------------------------------------------------------
# Two-tier chunk retrieval
# ---------------------------------------------------------------------------
def _get_brand_chunks(
    store:            "PolicyStore",
    pdf_name:         str,
    brand_name:       str,
    anchor_chunk_ids: List[str],
) -> str:
    """
    Tier 1: anchor chunks (from brand-detection stage 2) -- guaranteed relevant.
    Tier 2: brand-specific hybrid search fills remaining slots.
    Tier 3: common-query fill for any still-empty slots.
    Hard cap: PARAM_TOP_K chunks total.
    """
    seen:  set       = set()
    texts: List[str] = []

    # -- Tier 1: anchor chunks -----------------------------------------------
    if anchor_chunk_ids:
        try:
            fetched = store.col.get(
                ids=anchor_chunk_ids,
                include=["documents", "metadatas"],
            )
            for cid, doc, meta in zip(
                fetched["ids"], fetched["documents"], fetched["metadatas"]
            ):
                if meta.get("pdf") == pdf_name and cid not in seen:
                    seen.add(cid)
                    texts.append(doc)
        except Exception as exc:
            log.warning("Anchor fetch failed for %s: %s", brand_name, exc)

    # -- Tier 2: brand-specific hybrid search --------------------------------
    if len(texts) < PARAM_TOP_K:
        brand_query = (
            f"{brand_name} prior authorization criteria plaque psoriasis "
            f"step therapy reauthorization approval"
        )
        for r in store.hybrid_search(brand_query, top_k=PARAM_TOP_K):
            if r["metadata"]["pdf"] == pdf_name and r["chunk_id"] not in seen:
                seen.add(r["chunk_id"])
                texts.append(r["text"])
            if len(texts) >= PARAM_TOP_K:
                break

    # -- Tier 3: common parameter queries ------------------------------------
    for query in _COMMON_QUERIES:
        if len(texts) >= PARAM_TOP_K:
            break
        for r in store.hybrid_search(query, top_k=2):
            if r["metadata"]["pdf"] == pdf_name and r["chunk_id"] not in seen:
                seen.add(r["chunk_id"])
                texts.append(r["text"])
            if len(texts) >= PARAM_TOP_K:
                break

    return "\n\n---\n\n".join(texts[:PARAM_TOP_K])


# ---------------------------------------------------------------------------
# JSON + scoring helpers
# ---------------------------------------------------------------------------
def _parse_brand_json(raw: str, brand_name: str) -> Dict[str, Any]:
    try:
        start = raw.index("{")
        end   = raw.rindex("}") + 1
        return json.loads(raw[start:end])
    except (ValueError, json.JSONDecodeError):
        log.error("JSON parse failed for brand '%s':\n%s", brand_name, raw[:200])
        return {}


def _score_age(val: str) -> float:
    v = val.strip().lower()
    if v in ("na", "", "fda labelled age", "fda labeled age"):
        return 1.0
    m = re.search(r"(\d+)", v)
    if not m:
        return 0.7
    age = int(m.group(1))
    if age <= 6:   return 1.0
    if age <= 12:  return 0.9
    if age <= 18:  return 0.7
    return 0.4


def _score_steps(val: str) -> float:
    v = val.strip().lower()
    if v in ("na", "", "0"):  return 1.0
    m = re.search(r"(\d+)", v)
    if not m:                  return 1.0
    return max(0.0, 1.0 - int(m.group(1)) * 0.3)


def _score_duration(val: str) -> float:
    v = val.strip().lower()
    if v in ("na", "", "unspecified"):  return 0.5
    m = re.search(r"(\d+)", v)
    if not m:  return 0.5
    months = int(m.group(1))
    if months >= 12:  return 1.0
    if months >= 6:   return 0.7
    if months >= 3:   return 0.4
    return 0.2


def _score_yesno(val: str, yes_score: float = 0.3, no_score: float = 1.0) -> float:
    v = val.strip().lower()
    if v in ("yes", "y"):  return yes_score
    if v in ("no", "n"):   return no_score
    return 0.7


def _score_text_present(val: str) -> float:
    v = val.strip().lower()
    return 0.3 if v and v != "na" else 1.0


_WEIGHTS = {
    "Number of Steps through Brands":                   20,
    "Initial Authorization Duration(in-months)":        15,
    "TB Test required":                                 15,
    "Age":                                              10,
    "Number of Steps through Generic":                  10,
    "Step through-Phototherapy":                         5,
    "Step Therapy Requirements Documented in Policy":    5,
    "Reauthorization Required":                          5,
    "Reauthorization Duration(in-months)":               5,
    "Specialist Types":                                  4,
    "Reauthorization Requirements Documented in Policy": 3,
    "Quantity Limits":                                   3,
}


def compute_access_score(row: Dict[str, str]) -> int:
    scorers = {
        "Age":                                            lambda v: _score_age(v),
        "Step Therapy Requirements Documented in Policy": lambda v: _score_text_present(v),
        "Number of Steps through Brands":                 lambda v: _score_steps(v),
        "Number of Steps through Generic":                lambda v: _score_steps(v),
        "Step through-Phototherapy":                      lambda v: _score_yesno(v),
        "TB Test required":                               lambda v: _score_yesno(v, yes_score=0.3, no_score=1.0),
        "Initial Authorization Duration(in-months)":      lambda v: _score_duration(v),
        "Reauthorization Duration(in-months)":            lambda v: _score_duration(v),
        "Reauthorization Required":                       lambda v: _score_yesno(v),
        "Reauthorization Requirements Documented in Policy": lambda v: _score_text_present(v),
        "Specialist Types":                               lambda v: _score_text_present(v),
        "Quantity Limits":                                lambda v: _score_text_present(v),
    }
    return round(sum(_WEIGHTS[p] * scorers[p](row.get(p, "NA")) for p in _WEIGHTS))


def _flatten_row(filename: str, brand_result: Dict) -> Dict[str, str]:
    row = {"filename": filename, "brand": brand_result.get("brand", "")}
    for p in PARAMS:
        row[p] = str(brand_result.get(p, "NA"))
    row["access_score"] = str(compute_access_score(row))
    return row


# ---------------------------------------------------------------------------
# Core extraction -- one 70B LLM call per brand
# ---------------------------------------------------------------------------
def extract_params(
    md_path:    Path,
    brand_data: Dict[str, Any],
    store:      "PolicyStore",
    llm:        LLMClient,        # pass llm_70b here
) -> List[Dict[str, str]]:
    """
    For each brand: anchor chunks + hybrid search -> 70B LLM -> 12 params.
    RateLimiter inside LLMClient ensures Groq 12K TPM limit is respected.
    """
    pdf_name = md_path.stem + ".pdf"
    brands   = brand_data.get("brands_relevant_to_pso", [])
    if not brands:
        log.info("  No brands -- skipping")
        return []

    # Load already-processed (pdf, brand) pairs to skip them
    _done: set = set()
    if OUTPUT_CSV.exists():
        with open(OUTPUT_CSV, newline="", encoding="utf-8") as _f:
            for _r in csv.DictReader(_f):
                _done.add((_r.get("filename", ""), _r.get("brand", "")))

    rows: List[Dict[str, str]] = []
    for brand in tqdm(brands, desc=f"  {pdf_name}", unit="brand"):
        brand_name  = brand["brand"]
        preferred   = brand.get("preferred_status", "Unspecified")
        anchor_ids  = brand.get("anchor_chunk_ids", [])

        if (pdf_name, brand_name) in _done:
            log.info("  Skipping (already in CSV) -- %s", brand_name)
            continue

        log.info("  Extracting -- %s (%s)  anchors=%d", brand_name, preferred, len(anchor_ids))

        chunks = _get_brand_chunks(store, pdf_name, brand_name, anchor_ids)
        if not chunks.strip():
            log.warning("    No chunks retrieved for %s -- skipping", brand_name)
            continue
        log.info("    Chunks: %d chars -> 70B LLM", len(chunks))

        messages = [{
            "role": "user",
            "content": _PARAM_PROMPT.format(
                brand_name=brand_name,
                preferred_status=preferred,
                chunks=chunks,
            ),
        }]
        raw    = llm.complete(messages, temperature=0.0, max_tokens=1024)
        result = _parse_brand_json(raw, brand_name)
        if result:
            row = _flatten_row(pdf_name, result)
            rows.append(row)
            write_csv([row], OUTPUT_CSV, append=True)
            log.info("    Saved to CSV: %s", brand_name)

    log.info("  Total rows extracted: %d", len(rows))
    return rows


# ---------------------------------------------------------------------------
# CSV writer
# ---------------------------------------------------------------------------
def write_csv(rows: List[Dict], output_path: Path, append: bool = True) -> None:
    existing: set = set()
    if append and output_path.exists():
        with open(output_path, newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                existing.add((r.get("filename", ""), r.get("brand", "")))

    new_rows = [r for r in rows if (r["filename"], r["brand"]) not in existing]
    skipped  = len(rows) - len(new_rows)
    if skipped:
        log.info("  Skipped %d duplicate row(s) already in CSV", skipped)
    if not new_rows:
        log.info("  No new rows to write")
        return

    mode         = "a" if (append and output_path.exists()) else "w"
    write_header = not (append and output_path.exists())
    with open(output_path, mode, newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerows(new_rows)
    log.info("CSV updated -- %d new row(s) -> %s", len(new_rows), output_path)


print("Parameter extraction and access score defined.")


## 7. Run Pipeline

Full end-to-end pipeline for a single PDF:

1. PDF -> Markdown (Docling, CPU)
2. Chunk & Store (700/100, 384-dim bge-small)
3. Brand Detection (8B + hybrid search)
4. Parameter Extraction (70B + two-tier retrieval, rate-limited)
5. Write CSV


In [ ]:
def run_pipeline(pdf_path: Path) -> None:
    log.info("=" * 65)
    log.info("PDF  : %s", pdf_path.name)
    log.info("=" * 65)

    # ------------------------------------------------------------------
    # STEP 1 -- PDF -> Markdown
    # ------------------------------------------------------------------
    log.info("STEP 1 -- PDF -> Markdown")
    converter = build_converter()
    md_path   = convert_pdf(pdf_path, MARKDOWN_DIR, converter)
    if md_path is None:
        log.error("Markdown conversion failed -- aborting")
        return
    log.info("  Markdown: %s", md_path)

    # ------------------------------------------------------------------
    # STEP 2 -- Chunk & Store (ephemeral ChromaDB, deleted after run)
    # ------------------------------------------------------------------
    log.info("STEP 2 -- Chunk & Store  [bge-small-en-v1.5, 384-dim]")
    pdf_name = pdf_path.stem + ".pdf"
    md_text  = md_path.read_text(encoding="utf-8")
    chunks   = chunk_markdown(md_text, pdf_name)
    log.info("  Chunks produced: %d", len(chunks))

    tmp_dir = tempfile.mkdtemp(prefix="chroma_main_")
    try:
        store = PolicyStore(chroma_dir=Path(tmp_dir))
        store.add_chunks(chunks)

        # ------------------------------------------------------------------
        # STEP 3 -- Brand Detection  [8B]
        # ------------------------------------------------------------------
        log.info("STEP 3 -- Brand Detection  [%s]", GROQ_MODEL_8B)
        llm_8b     = LLMClient("8b")
        brand_data = detect_brands(md_path, store, llm_8b)

        has_pso = brand_data.get("policy_has_pso", "No")
        brands  = brand_data.get("brands_relevant_to_pso", [])
        log.info("  policy_has_pso : %s", has_pso)
        log.info("  brands found   : %s", [b["brand"] for b in brands])

        if has_pso != "Yes" or not brands:
            log.info("  No PsO brands -- nothing to extract")
            return

        # ------------------------------------------------------------------
        # STEP 4 -- Parameter Extraction  [70B]
        # ------------------------------------------------------------------
        log.info("STEP 4 -- Parameter Extraction  [%s]", GROQ_MODEL_70B)
        llm_70b = LLMClient("70b", fallback=llm_8b)
        rows    = extract_params(md_path, brand_data, store, llm_70b)

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    # ------------------------------------------------------------------
    # STEP 5 -- Summary
    # ------------------------------------------------------------------
    if rows:
        log.info("STEP 5 -- Done: %d brand(s) extracted and saved for %s", len(rows), pdf_path.name)
        for r in rows:
            log.info(
                "    %-30s  steps_brand=%-4s  reauth=%-4s  TB=%s  score=%s",
                r["brand"],
                r["Number of Steps through Brands"],
                r["Reauthorization Required"],
                r["TB Test required"],
                r["access_score"],
            )
    else:
        log.info("  No rows extracted for %s", pdf_path.name)

    log.info("LLM cache: %d entries", llm_8b.cache_size)


print("run_pipeline defined.")


## 8. Execute

Processes every PDF in `PDF_DIR` sequentially. Already-converted markdowns are skipped (Step 1 caches them). Already-extracted `(filename, brand)` pairs are skipped in the CSV.

In [59]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Set to None to process all PDFs, or a number like 5 to process only the first N.
MAX_PDFS = 10

pdfs = sorted(PDF_DIR.glob("*.pdf"))
if not pdfs:
    raise FileNotFoundError(f"No PDFs found in {PDF_DIR}")

if MAX_PDFS is not None:
    pdfs = pdfs[:MAX_PDFS]

print(f"Processing {len(pdfs)} PDF(s)  (MAX_PDFS={MAX_PDFS})")

# Step 1: convert all PDFs to markdown in parallel
print("\nStep 1: PDF -> Markdown (parallel)")
converter = build_converter()

def _convert(pdf):
    return pdf, convert_pdf(pdf, MARKDOWN_DIR, converter)

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(_convert, pdf): pdf for pdf in pdfs}
    md_map  = {}
    with tqdm(total=len(pdfs), desc="Converting PDFs", unit="pdf") as pbar:
        for fut in as_completed(futures):
            pdf, md = fut.result()
            md_map[pdf] = md
            pbar.update(1)

# Steps 2-5: brand detection + extraction sequentially
print("\nSteps 2-5: Brand detection + extraction")
for i, pdf in enumerate(tqdm(pdfs, desc="Processing PDFs", unit="pdf")):
    md_path = md_map.get(pdf)
    if md_path is None:
        log.error("Skipping %s — markdown conversion failed", pdf.name)
        continue
    print(f"\n[{i+1}/{len(pdfs)}] {pdf.name}")
    try:
        run_pipeline(pdf)
    except Exception as e:
        log.error("Pipeline failed for %s: %s", pdf.name, e)
        continue

print(f"\nAll done. Output: {OUTPUT_CSV}")


09:06:36  INFO     [skip]    148593-4960549.pdf — markdown already exists
09:06:36  INFO     [skip]    163262-4627017.pdf — markdown already exists


Processing 10 PDF(s)  (MAX_PDFS=10)

Step 1: PDF -> Markdown (parallel)


09:06:36  INFO     [skip]    167126-4508319.pdf — markdown already exists


Converting PDFs:   0%|          | 0/10 [00:00<?, ?pdf/s]

09:06:36  INFO     [skip]    176207-4867884.pdf — markdown already exists
09:06:36  INFO     [skip]    176806-5005129.pdf — markdown already exists
09:06:36  INFO     [skip]    176810-4889390.pdf — markdown already exists
09:06:36  INFO     [skip]    183953-4805567.pdf — markdown already exists
09:06:36  INFO     [skip]    187701-5050284.pdf — markdown already exists
09:06:36  INFO     [skip]    195158-4643510.pdf — markdown already exists
09:06:36  INFO     [skip]    195239-4641478.pdf — markdown already exists


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

09:07:35  INFO     Brand detection — 195239-4641478.pdf  (14 chunks, 54047 chars)
09:07:35  INFO     HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 413 Payload Too Large"
09:07:39  INFO     HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 413 Payload Too Large"
09:07:43  INFO     HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 413 Payload Too Large"
09:07:53  INFO     HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 413 Payload Too Large"
09:07:53  ERROR    Pipeline failed for 195239-4641478.pdf: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01ksx1axerf11s49n116kj108q` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 7768, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code


All done. Output: /Users/sauravpandey/Downloads/Develop/projects/Payer_policy/Final_Code/submission.csv
